In [ ]:
!pip install --upgrade synthcity
!pip show synthcity

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchtext torchmetrics torchao torchtune torchtuples
!pip install torch==2.4.0 torchvision==0.19.0 torchtext==0.18.0

In [ ]:
!pip install torchtuples
!pip uninstall numpy -y
!pip install --upgrade "numpy==2.0.0" "networkx>=3.2" 

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/dataset-ricoveri/dataset_ricoveri_encode_index.csv',
    sep=';',
    quotechar='"',
    encoding='utf-8',
    on_bad_lines='skip'
)

print(df.head())
print(df.columns)

In [ ]:
# stdlib
import sys
import warnings

warnings.filterwarnings("ignore")

# synthcity absolute
#import synthcity.logger as log

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split

cod_regs = df["COD_REG"].dropna().unique()

train_cod, test_cod = train_test_split(
    cod_regs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = df[df["COD_REG"].isin(train_cod)].copy()
test_df  = df[df["COD_REG"].isin(test_cod)].copy()

In [12]:
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from synthcity.plugins.core.dataloader import TimeSeriesSurvivalDataLoader


class RicoveriTimeSeriesDataloader:
    """
    Prepara i dati di ricovero per synthcity TimeSeriesDataLoader.
    """

    TEMPORAL_CONTINUOUS = [
        "eta",
        "qt_prest_Sum",
    ]

    TEMPORAL_BINARY = [
        "SHOCK",
        "CABG",
        "PTCA",
        "ICD"
    ]

    TEMPORAL_CATEGORICAL = [
        "class_prest",
    ]

    TIME_INDEX_COL = "time_days"

    STATIC_CONTINUOUS = [
        "MCS_score",
    ]

    STATIC_BINARY = [
        "sesso",
    ]

    STATIC_CATEGORICAL = [
        "gruppo",
    ]

    OUTCOME_EVENT_COL = "dec_intra"
    OUTCOME_TIME_COL = "death_time"

    def __init__(
        self,
        seq_len: int = 5,
        step: int = 5,
        max_windows_per_patient: int = 1,
        balance_deceased: bool = True,
        n_patients: int = 2000,
        random_seed: int = 42,
    ):
        self.seq_len = seq_len
        self.step = step
        self.max_windows = max_windows_per_patient
        self.balance_deceased = balance_deceased
        self.n_patients = n_patients
        self.seed = random_seed

        self.scaler_temp = MinMaxScaler()
        self.scaler_static = MinMaxScaler()
        self.encoder_temp = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.encoder_static = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

        self.imputer_temp_cont = SimpleImputer(strategy="median")
        self.imputer_temp_cat = SimpleImputer(strategy="most_frequent")
        self.imputer_static_cont = SimpleImputer(strategy="median")
        self.imputer_static_cat = SimpleImputer(strategy="most_frequent")

    def _select_patients(self, df: pd.DataFrame) -> pd.DataFrame:
        rng = np.random.default_rng(self.seed)

        eligible = [
            cod for cod, grp in df.groupby("COD_REG")
            if len(grp) > 2 and (grp["desc_studio_out"] != 2).any()
        ]

        deceased = [c for c in eligible if df[df["COD_REG"] == c]["dec_intra"].max() == 1]
        survived = [c for c in eligible if df[df["COD_REG"] == c]["dec_intra"].max() == 0]

        if self.balance_deceased:
            n_dec = min(len(deceased), self.n_patients // 2)
            n_surv = min(len(survived), self.n_patients - n_dec)
        else:
            n_dec = min(len(deceased), self.n_patients)
            n_surv = 0

        chosen_dec = rng.choice(deceased, size=n_dec, replace=False)
        chosen_surv = rng.choice(survived, size=n_surv, replace=False)
        chosen = np.concatenate([chosen_dec, chosen_surv])

        print(f"Pazienti selezionati → deceduti: {n_dec}, sopravvissuti: {n_surv}, totale: {len(chosen)}")
        return df[df["COD_REG"].isin(chosen)].copy()

    def _preprocess_temporal(self, data: pd.DataFrame, fit: bool = True) -> pd.DataFrame:

        if self.TEMPORAL_CONTINUOUS:
            cont = data[self.TEMPORAL_CONTINUOUS].copy()
            if fit:
                cont_imp = pd.DataFrame(
                    self.imputer_temp_cont.fit_transform(cont),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
                cont_scaled = pd.DataFrame(
                    self.scaler_temp.fit_transform(cont_imp),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
            else:
                cont_imp = pd.DataFrame(
                    self.imputer_temp_cont.transform(cont),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
                cont_scaled = pd.DataFrame(
                    self.scaler_temp.transform(cont_imp),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
            data[self.TEMPORAL_CONTINUOUS] = cont_scaled

        if self.TEMPORAL_BINARY:
            bin_imp = SimpleImputer(strategy="most_frequent")
            data[self.TEMPORAL_BINARY] = bin_imp.fit_transform(data[self.TEMPORAL_BINARY])
            data[self.TEMPORAL_BINARY] = data[self.TEMPORAL_BINARY].astype(int)

        if self.TEMPORAL_CATEGORICAL:
            cat = data[self.TEMPORAL_CATEGORICAL].copy()
            if fit:
                cat_imp = pd.DataFrame(
                    self.imputer_temp_cat.fit_transform(cat),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
                cat_enc = pd.DataFrame(
                    self.encoder_temp.fit_transform(cat_imp),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
            else:
                cat_imp = pd.DataFrame(
                    self.imputer_temp_cat.transform(cat),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
                cat_enc = pd.DataFrame(
                    self.encoder_temp.transform(cat_imp),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
            data[self.TEMPORAL_CATEGORICAL] = cat_enc

        return data

    def _preprocess_static(self, static: pd.DataFrame, fit: bool = True) -> pd.DataFrame:

        if self.STATIC_CONTINUOUS:
            cont = static[self.STATIC_CONTINUOUS].copy()
            if fit:
                cont_imp = pd.DataFrame(
                    self.imputer_static_cont.fit_transform(cont),
                    columns=self.STATIC_CONTINUOUS, index=static.index
                )
                static[self.STATIC_CONTINUOUS] = self.scaler_static.fit_transform(cont_imp)
            else:
                cont_imp = pd.DataFrame(
                    self.imputer_static_cont.transform(cont),
                    columns=self.STATIC_CONTINUOUS, index=static.index
                )
                static[self.STATIC_CONTINUOUS] = self.scaler_static.transform(cont_imp)

        if self.STATIC_BINARY:
            bin_imp = SimpleImputer(strategy="most_frequent")
            static[self.STATIC_BINARY] = bin_imp.fit_transform(static[self.STATIC_BINARY])
            static[self.STATIC_BINARY] = static[self.STATIC_BINARY].astype(int)

        if self.STATIC_CATEGORICAL:
            cat = static[self.STATIC_CATEGORICAL].copy()
            if fit:
                cat_imp = pd.DataFrame(
                    self.imputer_static_cat.fit_transform(cat),
                    columns=self.STATIC_CATEGORICAL, index=static.index
                )
                static[self.STATIC_CATEGORICAL] = self.encoder_static.fit_transform(cat_imp)
            else:
                cat_imp = pd.DataFrame(
                    self.imputer_static_cat.transform(cat),
                    columns=self.STATIC_CATEGORICAL, index=static.index
                )
                static[self.STATIC_CATEGORICAL] = self.encoder_static.transform(cat_imp)

        return static

    def _make_windows(
        self,
        group: pd.DataFrame,
        temporal_cols: List[str],
    ) -> Tuple[List[pd.DataFrame], List[np.ndarray]]:

        windows, times = [], []
        n = len(group)

        indices = range(0, max(1, n - self.seq_len + 1), self.step)
        chosen_indices = list(indices)[: self.max_windows]

        for start in chosen_indices:
            end = start + self.seq_len
            chunk = group.iloc[start:end]

            if len(chunk) < self.seq_len:
                pad_rows = self.seq_len - len(chunk)
                pad = pd.concat([chunk.iloc[[-1]]] * pad_rows, ignore_index=True)
                chunk = pd.concat([chunk, pad], ignore_index=True)

            windows.append(chunk[temporal_cols].reset_index(drop=True))
            times.append(chunk[self.TIME_INDEX_COL].values)

        return windows, times

    def inverse_static(self, static_df: pd.DataFrame) -> pd.DataFrame:
        df = static_df.copy()
    
        # CONTINUE
        if self.STATIC_CONTINUOUS:
            df[self.STATIC_CONTINUOUS] = self.scaler_static.inverse_transform(
                df[self.STATIC_CONTINUOUS]
            )
    
        # CATEGORICHE
        if self.STATIC_CATEGORICAL:
            df[self.STATIC_CATEGORICAL] = self.encoder_static.inverse_transform(
                df[self.STATIC_CATEGORICAL]
            )
    
        return df

    def inverse_temporal(self, windows: List[pd.DataFrame]) -> List[pd.DataFrame]:
        inv_windows = []
    
        for w in windows:
            w = w.copy()
    
            # CONTINUE
            if self.TEMPORAL_CONTINUOUS:
                w[self.TEMPORAL_CONTINUOUS] = self.scaler_temp.inverse_transform(
                    w[self.TEMPORAL_CONTINUOUS]
                )
    
            # CATEGORICHE
            if self.TEMPORAL_CATEGORICAL:
                w[self.TEMPORAL_CATEGORICAL] = self.encoder_temp.inverse_transform(
                    w[self.TEMPORAL_CATEGORICAL]
                )
    
            inv_windows.append(w)
    
        return inv_windows
    
    def load(
        self,
        df: pd.DataFrame,
        fit: bool = True,
    ) -> TimeSeriesSurvivalDataLoader:

        data = self._select_patients(df) if fit else df.copy()
        data = data.sort_values(["COD_REG", "time_step"]).reset_index(drop=True)

        temporal_cols = self.TEMPORAL_CONTINUOUS + self.TEMPORAL_BINARY + self.TEMPORAL_CATEGORICAL
        data = self._preprocess_temporal(data, fit=fit)

        all_windows, all_times, static_rows, T_list, E_list = [], [], [], [], []

        for cod, group in data.groupby("COD_REG"):
            if len(group) < 2:
                continue

            wins, times = self._make_windows(group, temporal_cols)

            for w, t in zip(wins, times):
                w = w.fillna(0)  # ✅ FIX TEMPORAL

                all_windows.append(w)
                all_times.append(t)

                static_row = group[
                    self.STATIC_CONTINUOUS + self.STATIC_BINARY + self.STATIC_CATEGORICAL
                ].iloc[-1]

                static_rows.append(static_row)

                E_list.append(int(group[self.OUTCOME_EVENT_COL].max()))
                T_list.append(float(group[self.OUTCOME_TIME_COL].max()))

        static_df = pd.DataFrame(static_rows).reset_index(drop=True)
        static_df = self._preprocess_static(static_df, fit=fit)

        static_df = static_df.fillna(0)  # ✅ FIX STATIC

        T = np.array(T_list, dtype=float)
        E = np.array(E_list, dtype=float)
        
        # ✅ FIX CRITICO
        T = np.nan_to_num(T, nan=0.0, posinf=0.0, neginf=0.0)
        E = np.nan_to_num(E, nan=0.0)
        
        E = E.astype(int)

        # ✅ DEBUG CHECK (opzionale)
        assert not np.isnan(static_df.values).any(), "NaN nelle static!"
        assert not any(w.isna().any().any() for w in all_windows), "NaN nelle temporal!"

        print(f"Finestre totali: {len(all_windows)}")
        print(f"Pazienti con evento (dec_intra=1): {E.sum()} / {len(E)}")

        loader = TimeSeriesSurvivalDataLoader(
            temporal_data=all_windows,
            observation_times=all_times,
            static_data=static_df,
            T=T,
            E=E,
            static_categorical_columns=self.STATIC_BINARY + self.STATIC_CATEGORICAL,
            temporal_categorical_columns=self.TEMPORAL_BINARY + self.TEMPORAL_CATEGORICAL,
        )

        return loader

In [ ]:
l = RicoveriTimeSeriesDataloader(
      seq_len=5,
      step=5,
      max_windows_per_patient=1,
      balance_deceased=True,   # ← FIX: include anche sopravvissuti
      n_patients=5000,
  )

loader_train = l.load(train_df, fit=True)

In [ ]:
loader_test = l.load(test_df, fit=False)

In [ ]:
# synthcity absolute
from synthcity.plugins import Plugins

syn_model = Plugins().get(
    "timegan",

    n_iter=3000,
    mode="Transformer",

    generator_n_layers_hidden=3,
    generator_n_units_hidden=256,

    discriminator_n_layers_hidden=3,
    discriminator_n_units_hidden=256,

    embedding_dim=32,

    generator_lr=1e-4,
    discriminator_lr=1e-4,

    enc_cluster=60,
    moments_penalty=20,
    embedding_penalty=10,
    #gamma_penalty=5,
)

In [ ]:
# synthcity absolute
from synthcity.plugins import Plugins

syn_model = Plugins().get(
    "timevae",

    n_iter=3000,
    mode="Transformer",

    encoder_n_layers_hidden=3,
    encoder_n_units_hidden=200,

    decoder_n_layers_hidden=3,
    decoder_n_units_hidden=200,

    decoder_batch_norm=True,
    decoder_dropout=0.05,
    decoder_residual=True,

    latent_dim=64,
)

In [20]:
import os

os.listdir("/kaggle/working")
os.listdir("/kaggle/working/workspace")
os.makedirs("/kaggle/working/workspace", exist_ok=True)

In [ ]:
syn_model.fit(loader)

In [61]:
from synthcity.utils.serialization import save, load
from synthcity.utils.serialization import save_to_file, load_from_file

save_to_file('/kaggle/working/workspace/timegan_1500p_transformer_2g_2d_units_100_1500ep_norm.pkl', syn_model)

In [ ]:
synthetic = syn_model.generate(count=1500)

In [ ]:
from synthcity.metrics.eval_statistical import StatisticalEvaluator, ChiSquaredTest

metric = ChiSquaredTest()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_statistical import JensenShannonDistance

metric = JensenShannonDistance()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_statistical import InverseKLDivergence

metric = InverseKLDivergence()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_statistical import PRDCScore

metric = PRDCScore()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_detection import SyntheticDetectionMLP

metric = SyntheticDetectionMLP()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import kAnonymization

metric = kAnonymization()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import lDiversityDistinct

metric = lDiversityDistinct()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import DeltaPresence

metric = DeltaPresence()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import kMap

metric = kMap()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import IdentifiabilityScore

metric = IdentifiabilityScore()
print(metric.evaluate(loader_test,synthetic))

In [15]:
import numpy as np
import pandas as pd

def _to_dataframe(loader):
    """
    Converte un GenericDataLoader di synthcity in pandas.DataFrame.
    In synthcity di solito è .dataframe() oppure .to_pandas().
    Proviamo entrambe.
    """
    if hasattr(loader, "dataframe"):
        return loader.dataframe()
    if hasattr(loader, "to_pandas"):
        return loader.to_pandas()
    raise TypeError("Il dataloader non espone dataframe() o to_pandas().")

In [16]:
df_r = _to_dataframe(loader_train).copy()
df_s = _to_dataframe(synthetic).copy()

In [20]:
import os
import matplotlib.pyplot as plt

def plot_feature_distributions(X_real, X_syn, feature_names=None, bins=30, output_dir="/kaggle/working/workspace/feature_plots"):
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Convert DataFrame to numpy if needed
    if hasattr(X_real, "values"):
        X_real = X_real.values
    if hasattr(X_syn, "values"):
        X_syn = X_syn.values

    n_features = X_real.shape[1]
    
    for i in range(n_features):
        plt.figure()
        
        plt.hist(X_real[:, i], bins=bins, alpha=0.5, density=True, label="Real")
        plt.hist(X_syn[:, i], bins=bins, alpha=0.5, density=True, label="Synthetic")
        
        if feature_names:
            title = feature_names[i]
            plt.title(f"Feature: {title}")
            filename = f"{title}.png"
        else:
            title = f"feature_{i}"
            plt.title(f"Feature {i}")
            filename = f"{title}.png"
            
        plt.legend()
        
        # Save the figure
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, bbox_inches="tight", dpi=300)
        
        plt.show()
        plt.close()

In [ ]:
import matplotlib.pyplot as plt

plot_feature_distributions(df_r, synthetic, feature_names)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_combined = np.vstack([df_r, df_s])
X_scaled = scaler.fit_transform(X_combined)

X_real_scaled = X_scaled[:len(df_r)]
X_syn_scaled = X_scaled[len(df_s):]

In [ ]:
from sklearn.manifold import TSNE

# Combiniamo
X_combined = np.vstack([X_real_scaled, X_syn_scaled])

labels = np.array(
    ["Real"] * len(X_real_scaled) + 
    ["Synthetic"] * len(X_syn_scaled)
)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_combined)

# Plot
plt.figure()
for lab in ["Real", "Synthetic"]:
    idx = labels == lab
    plt.scatter(X_tsne[idx, 0], X_tsne[idx, 1], label=lab, alpha=0.6)

plt.legend()
plt.title("t-SNE: Real vs Synthetic")
filepath = os.path.join('/kaggle/working/workspace/feature_plots', 'tsne_plot.png')
plt.savefig(filepath, bbox_inches="tight", dpi=300)
plt.show()

In [23]:
d = loader_train.dataframe()

In [24]:
d_s = synthetic.dataframe()

In [ ]:
temporal_cols = [
    "age", "class_prest", "qt_prest_Sum", "SHOCK", "CABG", "PTCA", "ICD"
]

static_cols = [
    "sex", "group", "MCS_score"
]

temporal_cols_seq = [f"seq_temporal_{c}" for c in temporal_cols]
static_cols_seq = [f"seq_static_{c}" for c in static_cols]

temporal_data = []
static_data = []

for seq_id, group in d.groupby("seq_id"):
    group = group.sort_values("seq_time_id")

    # temporal
    temp = group[temporal_cols_seq].copy()
    temp.columns = temporal_cols
    temporal_data.append(temp.reset_index(drop=True))

    # static (una sola riga)
    stat = group[static_cols_seq].iloc[0]
    static_data.append(stat.values)

static_data = pd.DataFrame(static_data, columns=static_cols)

# 🔁 inverse transform
temporal_denorm = loader_pre.inverse_transform_temporal(temporal_data)
static_denorm = loader_pre.inverse_transform_static(static_data)

In [ ]:
temporal_cols = [
    "age", "class_prest", "qt_prest_Sum", "SHOCK", "CABG", "PTCA", "ICD"
]

static_cols = [
    "sex", "group", "MCS_score"
]

temporal_cols_seq = [f"seq_temporal_{c}" for c in temporal_cols]
static_cols_seq = [f"seq_static_{c}" for c in static_cols]

temporal_data = []
static_data = []

for seq_id, group in d_s.groupby("seq_id"):
    group = group.sort_values("seq_time_id")

    # temporal
    temp = group[temporal_cols_seq].copy()
    temp.columns = temporal_cols
    temporal_data.append(temp.reset_index(drop=True))

    # static (una sola riga)
    stat = group[static_cols_seq].iloc[0]
    static_data.append(stat.values)

static_data = pd.DataFrame(static_data, columns=static_cols)

# 🔁 inverse transform
temporal_denorm_s = l.inverse_transform_temporal(temporal_data)
static_denorm_s = l.inverse_transform_static(static_data)

In [57]:
import pandas as pd

def build_long_df(temporal_list):
    rows = []
    for seq_id, temp_df in enumerate(temporal_list):
        df = temp_df.copy()
        df["seq_id"] = seq_id
        df["time_days"] = range(len(df))
        rows.append(df)
    return pd.concat(rows, ignore_index=True)

df_real_plot = build_long_df(temporal_denorm)
df_synth_plot = build_long_df(temporal_denorm_s)

In [58]:
import pandas as pd

# dati reali cumulativi
df_real_cum = pd.concat(temporal_denorm, ignore_index=True)

# dati sintetici cumulativi
df_synth_cum = pd.concat(temporal_denorm_s, ignore_index=True)

In [ ]:
import os
import matplotlib.pyplot as plt

# crea la cartella di output (se non esiste)
output_dir = "/kaggle/working/workspace/output_plots"
os.makedirs(output_dir, exist_ok=True)

temporal_vars = ["eta", "class_prest", "qt_prest_Sum", "SHOCK", "CABG", "PTCA", "ICD"] 

for var in temporal_vars: 
    #filepath = os.path.join("output_dir", filename) 

    filename = f"{var}_distribution.png"
    filepath = os.path.join(output_dir, filename)

    # salva figura
    plt.savefig(filepath, bbox_inches="tight", dpi=300)
    plt.savefig(filepath, bbox_inches="tight", dpi=300) 
    plt.figure(figsize=(6,4)) 
    plt.hist(df_real_cum[var], bins=30, alpha=0.5, label="Reale", density=True) 
    plt.hist(df_synth_cum[var], bins=30, alpha=0.5, label="Sintetico", density=True) 
    plt.xlabel(var) 
    plt.ylabel("Densità") 
    plt.title(f"Distribuzione cumulativa di {var}") 
    plt.legend() 
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

static_vars = ["sesso", "gruppo", "MCS_score"]
output_dir = "/kaggle/working/workspace/output_plots"

for var in static_vars:
    plt.figure(figsize=(6,4))

    # se è numerica
    if pd.api.types.is_numeric_dtype(static_denorm[var]):
        plt.hist(static_denorm[var], bins=30, alpha=0.5, label="Reale", density=True)
        plt.hist(static_denorm_s[var], bins=30, alpha=0.5, label="Sintetico", density=True)
    else:
        # se categorica, facciamo bar plot delle frequenze
        real_counts = static_denorm[var].value_counts(normalize=True)
        synth_counts = static_denorm_s[var].value_counts(normalize=True)
        all_categories = sorted(set(real_counts.index) | set(synth_counts.index))
        plt.bar([c for c in all_categories], [real_counts.get(c,0) for c in all_categories], alpha=0.5, label="Reale")
        plt.bar([c for c in all_categories], [synth_counts.get(c,0) for c in all_categories], alpha=0.5, label="Sintetico")

    filename = f"{var}_distribution.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, bbox_inches="tight", dpi=300)

    # salva figura
    plt.xlabel(var)
    plt.ylabel("Densità / Frequenza")
    plt.title(f"Distribuzione cumulativa {var}: reale vs sintetico")
    plt.legend()
    plt.show()